In [ ]:
# =============================================================================
# 0. INSTALL DEPENDENCIES & UNZIP
# =============================================================================

!pip install scikit-learn matplotlib tqdm pillow torchvision -q
!pip install -q --upgrade torchao





In [ ]:
# =============================================================================
# 1. IMPORTS
# =============================================================================

import os
import shutil
import random
import json
from collections import defaultdict, Counter
from io import BytesIO
import base64

import numpy as np
from PIL import Image, UnidentifiedImageError
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torchvision.transforms import functional as TF
import torchvision.models as models

from peft import LoraConfig, get_peft_model
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_curve,
    average_precision_score,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from sklearn.metrics import roc_curve, auc
import umap
import plotly.graph_objects as go
from IPython.display import display, HTML
from tqdm import tqdm
import time

In [ ]:
# =============================================================================
# 2. MOUNT GOOGLE DRIVE AND UNZIP DIRECTORY
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')

# !unzip /content/drive/MyDrive/final_dataset.zip -d /content/


In [ ]:
# =============================================================================
# 3. CONFIGURATION
# =============================================================================
# --- MODE ---
# Choose 'flies' or 'maggots' to select which subset to train on.
MODE = 'maggots'

device     = "cuda" if torch.cuda.is_available() else "cpu"


DATA_DIR   = "/content/final_dataset/"

PLOT_DIR   = "/content/drive/MyDrive/EffNet_plots"
MODEL_DIR  = "/content/drive/MyDrive/EffNet_models"
os.makedirs(PLOT_DIR,  exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)
print(f"Device     : {device}")
print(f"Plot dir   : {PLOT_DIR}")
print(f"Model dir  : {MODEL_DIR}")


# --- AUGMENTATIONS ---
# Add or remove augmentations as needed. Each entry is a function that takes a PIL image and returns a PIL image.
AUGMENTATIONS = [
    # lambda img: TF.rotate(img, 90),
    # lambda img: TF.hflip(TF.rotate(img, 180)),
]

# --- TRAINING HYPERPARAMETERS ---
BATCH_SIZE        = 16
EPOCHS            = 10
LEARNING_RATE     = 1e-4
WEIGHT_DECAY      = 0.01
TRIPLET_MARGIN    = 0.2

# --- LORA HYPERPARAMETERS ---
LORA_RANK         = 8
LORA_ALPHA        = 16
LORA_DROPOUT      = 0.1

# --- DATASET PARAMETERS ---
MIN_IMAGES             = 423    # minimum original images required for a class to be included
TRIAL_IMAGES_PER_CLASS = 423    # number of originals sampled per class for the trial dataset

In [ ]:
# =============================================================================
# 4. SAFE IMAGE FOLDER  (skips corrupt files automatically)
# =============================================================================

class SafeImageFolder(ImageFolder):
    def __init__(self, root, transform=None, verify=True):
        super().__init__(root, transform=transform)

        if verify:
            print(f"Verifying {len(self.samples)} images...")
            valid_samples = []
            for path, label in self.samples:
                try:
                    with Image.open(path) as img:
                        img.verify()
                    valid_samples.append((path, label))
                except (UnidentifiedImageError, OSError, Exception):
                    print(f"  Removing corrupt image: {path}")

            self.samples = valid_samples
            self.imgs    = valid_samples
            removed      = len(self.imgs) - len(valid_samples)  # already equal after assign
            print(f"  Dataset ready: {len(valid_samples)} valid images")

    def __getitem__(self, index):
        path, label = self.samples[index]
        sample = self.loader(path)
        if self.transform is not None:
            sample = self.transform(sample)
        return sample, label


In [ ]:
# =============================================================================
# 5. LOAD EFFICIENTNET-B0 BACKBONE
# =============================================================================
#
# We remove the final classifier so the network outputs raw feature vectors
# (1280-dim from the AdaptiveAvgPool output of EfficientNet-B0).
# These are our embeddings — equivalent to encode_image() in the BioCLIP nb.
#
# Preprocessing uses the same ImageNet mean/std that EfficientNet was
# pretrained with.

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225]),
])

# --- Build encoder ---
_effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)

class EfficientNetEncoder(nn.Module):
    """EfficientNet-B0 with classifier replaced by an L2-normalised embedding head."""
    def __init__(self, embed_dim=512, freeze_backbone=True):
        super().__init__()
        self.features = _effnet.features          # convolutional backbone
        self.pool     = _effnet.avgpool           # AdaptiveAvgPool2d → (B, 1280, 1, 1)
        backbone_dim  = _effnet.classifier[1].in_features  # 1280

        # Projection head: 1280 → embed_dim  (keeps embedding size comparable to BioCLIP's 512)
        self.projection = nn.Sequential(
            nn.Flatten(),
            nn.Linear(backbone_dim, embed_dim),
            nn.BatchNorm1d(embed_dim),
        )

        if freeze_backbone:
            for p in self.features.parameters():
                p.requires_grad = False
            print("Backbone frozen — only projection head is trainable.")
        else:
            print("Full model is trainable.")

    def encode_image(self, x):
        """Drop-in replacement for open_clip model.encode_image(x)."""
        x = self.features(x)
        x = self.pool(x)
        x = self.projection(x)
        return x   # caller is responsible for L2-normalising if desired

    def forward(self, x):
        return self.encode_image(x)


EMBED_DIM = 512
model = EfficientNetEncoder(embed_dim=EMBED_DIM, freeze_backbone=True).to(device)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)")
print(f"\nModel architecture:")
print(model)


In [ ]:
# =============================================================================
# 6. LOAD DATASET
# =============================================================================

dataset     = SafeImageFolder(DATA_DIR, transform=preprocess)
class_names = dataset.classes
num_classes = len(class_names)

corrected_names = [CORRECTED_NAMES.get(name, name) for name in class_names]

print(f"\nDataset: {len(dataset)} images across {num_classes} classes")
print(f"\nClass name mapping (folder -> display):")
for folder, corrected in zip(class_names, corrected_names):
    marker = "  " if folder == corrected else "* "
    print(f"  {marker}{folder!r:45} -> {corrected!r}")


In [ ]:
# =============================================================================
# 7. DATASET STATISTICS
# =============================================================================

from collections import defaultdict

print(f"Dataset path: {DATA_DIR}")
print(f"Total images: {len(dataset)}")
print(f"\nSamples per class:")

samples_by_class = defaultdict(list)
for path, label in dataset.samples:
    samples_by_class[label].append(path)

for label in sorted(samples_by_class.keys()):
    print(f"  [{label:2d}] {class_names[label]:45s}  {len(samples_by_class[label])} images")


In [ ]:
# Cell 8 — Data augmentation
# Generates augmented versions per image based on AUGMENTATIONS defined in CONFIG.
# Only processes fly or maggot classes depending on MODE.
# Output is saved to AUG_DIR and only needs to be run once per MODE.

# Filter classes by MODE
eligible_classes = defaultdict(list)
for path, label in dataset.samples:
    class_name = dataset.classes[label].lower()
    if MODE == 'maggots' and 'maggot' in class_name:
        eligible_classes[label].append((path, label))
    elif MODE == 'flies' and 'maggot' not in class_name:
        eligible_classes[label].append((path, label))

# Save originals + augmentations to AUG_DIR
total_saved = 0
for label, samples in sorted(eligible_classes.items()):
    class_name = dataset.classes[label]
    out_dir    = os.path.join(AUG_DIR, class_name)
    os.makedirs(out_dir, exist_ok=True)

    for path, _ in tqdm(samples, desc=class_name):
        img = Image.open(path).convert('RGB')
        img = img.resize((512, 512), Image.LANCZOS)
        img.save(os.path.join(out_dir, os.path.basename(path)))
        total_saved += 1

        for i, aug_fn in enumerate(AUGMENTATIONS):
            aug_img = aug_fn(img)
            fname   = f"{os.path.splitext(os.path.basename(path))[0]}_aug{i+1}.jpg"
            aug_img.save(os.path.join(out_dir, fname))
            total_saved += 1

print(f"Done. {total_saved} images saved to: {AUG_DIR}")

In [ ]:
# =============================================================================
# 9. LOAD AUGMENTED DATASET & BUILD TRIAL SAMPLES
# =============================================================================

aug_dataset = SafeImageFolder(AUG_DIR, transform=preprocess, verify=False)
print(f"Augmented dataset: {len(aug_dataset)} images across {len(aug_dataset.classes)} classes")

originals_by_class     = defaultdict(list)
augmentations_by_class = defaultdict(list)

for path, label in aug_dataset.samples:
    if '_aug' in os.path.basename(path):
        augmentations_by_class[label].append((path, label))
    else:
        originals_by_class[label].append((path, label))

# Only include classes with at least MIN_IMAGES originals
eligible_classes = {
    label: samples for label, samples in originals_by_class.items()
    if len(samples) >= MIN_IMAGES
}

print(f"\nEligible classes: {len(eligible_classes)}")

random.seed(42)
trial_samples = []

for label in sorted(eligible_classes.keys()):
    origs = originals_by_class[label]
    augs  = augmentations_by_class[label]

    random.shuffle(origs)
    random.shuffle(augs)

    combined = origs + augs
    selected = combined[:TRIAL_IMAGES_PER_CLASS]
    trial_samples.extend(selected)

print(f"\nTrial dataset: {len(trial_samples)} samples")
for label in sorted(set(lbl for _, lbl in trial_samples)):
    n = sum(1 for _, l in trial_samples if l == label)
    print(f"  [{label:2d}] {aug_dataset.classes[label]:45s}  {n}")

In [ ]:
# Cell 9b. Define Unseen folder

# 1. Configuration
# Assuming AUG_DIR is already defined as "/content/drive/MyDrive/vit_16_augmente"
# Define your UNSEEN_DIR alongside it
UNSEEN_DIR = os.path.join(os.path.dirname(AUG_DIR), "unseen_efficientnetb0_16")

# Make sure the unseen directory exists
os.makedirs(UNSEEN_DIR, exist_ok=True)

moved_classes = []
kept_classes = []

print(f"Scanning classes in: {AUG_DIR}...\n")

# 2. Iterate through each class folder in the augmented directory
for class_name in sorted(os.listdir(AUG_DIR)):
    class_path = os.path.join(AUG_DIR, class_name)

    # Ensure it's a directory
    if os.path.isdir(class_path):
        num_images = len(os.listdir(class_path))

        # 3. Check threshold and route
        if num_images < MIN_IMAGES:
            # Move the entire folder to UNSEEN_DIR
            destination_path = os.path.join(UNSEEN_DIR, class_name)
            shutil.move(class_path, destination_path)
            moved_classes.append((class_name, num_images))
            print(f" 📦 MOVED: {class_name} ({num_images} images) -> unseen")
        else:
            kept_classes.append((class_name, num_images))
            print(f" ✅ KEPT:  {class_name} ({num_images} images) -> augmented")

# 4. Summary
print(f"\n--- SUMMARY (Threshold: {MIN_IMAGES}) ---")
print(f"Classes kept for training: {len(kept_classes)}")
print(f"Classes moved to unseen:   {len(moved_classes)}")
print(f"Unseen directory: {UNSEEN_DIR}")

In [ ]:
# =============================================================================
# 10. TRAIN / VAL / TEST SPLIT  (80-10-10, stratified)
# =============================================================================

trial_by_class = defaultdict(list)
for path, label in trial_samples:
    trial_by_class[label].append((path, label))

random.seed(42)
train_samples = []
val_samples   = []
test_samples  = []

for label, samples in sorted(trial_by_class.items()):
    random.shuffle(samples)
    n         = len(samples)
    train_end = int(0.8 * n)
    val_end   = int(0.9 * n)
    train_samples.extend(samples[:train_end])
    val_samples  .extend(samples[train_end:val_end])
    test_samples .extend(samples[val_end:])

print(f"Train: {len(train_samples)}  |  Val: {len(val_samples)}  |  Test: {len(test_samples)}")


In [ ]:
# =============================================================================
# 11. TRIPLET DATASET & DATALOADER
# =============================================================================

class TripletDataset(Dataset):
    """
    For each anchor image, randomly samples a positive (same class)
    and negative (different class) to form a triplet.
    Identical interface to the BioCLIP notebook version.
    """
    def __init__(self, samples, transform=None):
        self.samples   = samples
        self.transform = transform
        self.by_class  = defaultdict(list)
        for path, label in samples:
            self.by_class[label].append(path)
        self.labels = list(self.by_class.keys())

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        anchor_path, anchor_label = self.samples[idx]

        # Positive: same class, different image
        pos_pool = [p for p in self.by_class[anchor_label] if p != anchor_path]
        pos_path = random.choice(pos_pool) if pos_pool else anchor_path

        # Negative: random different class
        neg_label = random.choice([l for l in self.labels if l != anchor_label])
        neg_path  = random.choice(self.by_class[neg_label])

        def load(p):
            img = Image.open(p).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img

        return load(anchor_path), load(pos_path), load(neg_path), anchor_label


train_dataset = TripletDataset(train_samples, transform=preprocess)
val_dataset   = TripletDataset(val_samples,   transform=preprocess)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train loader: {len(train_loader)} batches")
print(f"Val   loader: {len(val_loader)}   batches")


In [ ]:
# =============================================================================
# 14. EMBEDDING EXTRACTION HELPER
# =============================================================================
#
# Mirrors BioCLIP notebook's extract_embeddings().
# Embeddings are L2-normalised before return.

class EvalDataset(Dataset):
    """Simple image+label dataset — no triplet sampling."""
    def __init__(self, samples, transform):
        self.samples   = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        return self.transform(img), label


def extract_embeddings(samples, transform, model, device, batch_size=32):
    dataset = EvalDataset(samples, transform)
    loader  = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    all_embeddings, all_labels = [], []

    model.eval()
    with torch.no_grad():
        for imgs, labels in tqdm(loader, desc="Extracting embeddings"):
            imgs = imgs.to(device)
            emb  = model.encode_image(imgs)
            emb  = emb / emb.norm(dim=-1, keepdim=True)   # L2 normalise
            all_embeddings.append(emb.cpu().numpy())
            all_labels    .append(labels.numpy())

    return np.concatenate(all_embeddings), np.concatenate(all_labels)


print("extract_embeddings() ready.")


In [ ]:
# ==================================================================================
# BASELINE CNN TEST (skip this cell if there is no need for the baseline evaluation)
# ==================================================================================

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt

# Extract embeddings (no fine-tuning, base EfficientNet-B0)
print("Extracting val embeddings...")
val_embeddings, val_labels = extract_embeddings(val_samples, preprocess, model, device)

print("Extracting test embeddings...")
test_embeddings, test_labels = extract_embeddings(test_samples, preprocess, model, device)

# KNN classification (val as train, test as test)
print("\nRunning KNN classifier...")
knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(val_embeddings, val_labels)
test_preds = knn.predict(test_embeddings)

# Accuracy
overall_acc = np.mean(test_preds == test_labels)
print(f"\nBaseline Accuracy (untrained EfficientNet-B0): {overall_acc*100:.1f}%")

# Classification report
target_names = [aug_dataset.classes[i] for i in sorted(np.unique(test_labels))]
print("\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=target_names))

# Confusion matrix
fig, ax = plt.subplots(figsize=(12, 10))
cm = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title("Confusion Matrix — Baseline EfficientNet-B0 (untrained)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"baseline_confusion_matrix_cnn_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()

# t-SNE
print("\nComputing t-SNE...")
all_embeddings_combined = np.concatenate([val_embeddings, test_embeddings], axis=0)
all_labels_combined     = np.concatenate([val_labels, test_labels], axis=0)
all_splits              = np.array(['val'] * len(val_embeddings) + ['test'] * len(test_embeddings))

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
embeddings_2d = tsne.fit_transform(all_embeddings_combined)

fig, ax = plt.subplots(figsize=(14, 10))
colors = plt.colormaps['tab20'].resampled(len(np.unique(all_labels_combined)))
unique_labels = sorted(np.unique(all_labels_combined))

for i, label in enumerate(unique_labels):
    for split, marker, size in [('val', 'o', 40), ('test', '*', 120)]:
        mask = (all_labels_combined == label) & (all_splits == split)
        ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   color=colors(i), marker=marker, s=size, alpha=0.6,
                   label=aug_dataset.classes[label] if split == 'val' else None)

ax.set_title("t-SNE — Baseline EfficientNet-B0 (untrained)\n(circles = val, stars = test)", fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"baseline_tsne_cnn_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()

print("Done. Plots saved to Drive.")

In [ ]:
# =============================================================================
# 12. FINE-TUNE SETUP
# =============================================================================
#
# Phase 1 (default at model creation): backbone frozen, only projection head trains.
# Phase 2 (optional, run this cell after initial training): unfreeze last 2 blocks.

# Unfreeze the last 2 MBConv blocks of EfficientNet-B0 features (indices 7 & 8)
for p in model.features[7:].parameters():
    p.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable params after partial unfreeze: {trainable:,} / {total:,}  ({100*trainable/total:.1f}%)")


In [ ]:
# =============================================================================
# 13. TRIPLET LOSS WITH SEMI-HARD MINING
# =============================================================================
#
# Direct port from the BioCLIP notebook — no changes needed here.

class TripletLossWithMining(nn.Module):
    """
    Triplet loss with semi-hard negative mining.

    For each triplet (anchor, positive, negative):
      Easy:      d(a,n) > d(a,p) + margin  -> loss = 0
      Semi-hard: d(a,n) > d(a,p) but < d(a,p) + margin  -> best learning signal
      Hard:      d(a,n) < d(a,p)  -> unstable early on
    """
    def __init__(self, margin=0.2):
        super().__init__()
        self.margin = margin

    def forward(self, anchor, positive, negative):
        d_pos = torch.sum((anchor - positive) ** 2, dim=1)
        d_neg = torch.sum((anchor - negative) ** 2, dim=1)

        loss_raw = d_pos - d_neg + self.margin

        # Classify triplets
        easy      = (d_neg > d_pos + self.margin)
        hard      = (d_neg < d_pos)
        semi_hard = (~easy & ~hard)

        # Only back-prop on semi-hard triplets; use all for loss value tracking
        semi_hard_losses = loss_raw[semi_hard]
        if len(semi_hard_losses) > 0:
            loss = semi_hard_losses.clamp(min=0).mean()
        else:
            loss = loss_raw.clamp(min=0).mean()  # fall-back

        stats = {
            "easy":      easy.sum().item(),
            "semi_hard": semi_hard.sum().item(),
            "hard":      hard.sum().item(),
        }
        return loss, stats


triplet_loss_fn = TripletLossWithMining(margin=0.2)
print("TripletLossWithMining ready.")


In [ ]:
# =============================================================================
# 14. PER-EPOCH KNN ACCURACY HELPER
# =============================================================================

def evaluate(eval_samples, train_samples, model, device):
    """Extract embeddings and return KNN top-1 accuracy (k=5, cosine)."""
    train_emb, train_lbl = extract_embeddings(train_samples, preprocess, model, device)
    eval_emb,  eval_lbl  = extract_embeddings(eval_samples,  preprocess, model, device)
    knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
    knn.fit(train_emb, train_lbl)
    preds = knn.predict(eval_emb)
    return float(np.mean(preds == eval_lbl))


def compute_val_loss(samples, model, device):
    """Average triplet loss on val set."""
    val_ds  = TripletDataset(samples, transform=preprocess)
    val_ld  = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for anchor, positive, negative, _ in val_ld:
            anchor, positive, negative = anchor.to(device), positive.to(device), negative.to(device)
            a_emb = model.encode_image(anchor);   a_emb = a_emb / a_emb.norm(dim=-1, keepdim=True)
            p_emb = model.encode_image(positive); p_emb = p_emb / p_emb.norm(dim=-1, keepdim=True)
            n_emb = model.encode_image(negative); n_emb = n_emb / n_emb.norm(dim=-1, keepdim=True)
            loss, _ = triplet_loss_fn(a_emb, p_emb, n_emb)
            total_loss += loss.item()
    return total_loss / max(len(val_ld), 1)


In [ ]:
# =============================================================================
# 15. TRAINING LOOP
# =============================================================================

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-4,
    weight_decay=0.01,
)

EPOCHS        = 10
best_val_loss = float('inf')

history = {
    "train_loss":    [],
    "val_loss":      [],
    "train_acc":     [],
    "val_acc":       [],
    "easy_pct":      [],
    "semi_hard_pct": [],
    "hard_pct":      [],
}

start_time = time.time()

for epoch in range(EPOCHS):
    model.train()
    epoch_loss  = 0.0
    epoch_stats = {"easy": 0, "semi_hard": 0, "hard": 0}

    for anchor, positive, negative, labels in tqdm(
            train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        anchor   = anchor  .to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        a_emb = model.encode_image(anchor);   a_emb = a_emb / a_emb.norm(dim=-1, keepdim=True)
        p_emb = model.encode_image(positive); p_emb = p_emb / p_emb.norm(dim=-1, keepdim=True)
        n_emb = model.encode_image(negative); n_emb = n_emb / n_emb.norm(dim=-1, keepdim=True)

        loss, stats = triplet_loss_fn(a_emb, p_emb, n_emb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        for k in epoch_stats:
            epoch_stats[k] += stats[k]

    # ── Metrics ──────────────────────────────────────────────────────────────
    avg_train_loss = epoch_loss / len(train_loader)
    avg_val_loss   = compute_val_loss(val_samples, model, device)
    total_triplets = sum(epoch_stats.values())

    train_acc = evaluate(train_samples, train_samples, model, device)
    val_acc   = evaluate(val_samples,   train_samples, model, device)

    history["train_loss"]   .append(avg_train_loss)
    history["val_loss"]     .append(avg_val_loss)
    history["train_acc"]    .append(train_acc)
    history["val_acc"]      .append(val_acc)
    history["easy_pct"]     .append(100 * epoch_stats["easy"]      / max(total_triplets, 1))
    history["semi_hard_pct"].append(100 * epoch_stats["semi_hard"] / max(total_triplets, 1))
    history["hard_pct"]     .append(100 * epoch_stats["hard"]      / max(total_triplets, 1))

    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")
    print(f"  Train Acc:  {train_acc*100:.1f}% | Val Acc:  {val_acc*100:.1f}%")
    print(f"  Easy: {history['easy_pct'][-1]:.1f}%  "
          f"Semi-hard: {history['semi_hard_pct'][-1]:.1f}%  "
          f"Hard: {history['hard_pct'][-1]:.1f}%")

    # Save best model
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        save_path = os.path.join(MODEL_DIR, f"efficientnet_b0_triplet_best_{MODE}.pt")
        torch.save(model.state_dict(), save_path)
        print(f"  ✓ Saved best model  (val loss: {best_val_loss:.4f})")

elapsed = time.time() - start_time
print(f"\nTraining complete in {elapsed/60:.1f} min.  Best val loss: {best_val_loss:.4f}")


In [ ]:
# =============================================================================
# 16. EXTRACT & SAVE EMBEDDINGS
# =============================================================================
#
# Embeddings are saved to Drive as .npz files so they can be reloaded later
# without re-running the model (useful for rapid KNN / t-SNE iteration).

print("Extracting train embeddings...")
train_embeddings, train_labels = extract_embeddings(train_samples, preprocess, model, device)

print("Extracting val embeddings...")
val_embeddings, val_labels = extract_embeddings(val_samples, preprocess, model, device)

print("Extracting test embeddings...")
test_embeddings, test_labels = extract_embeddings(test_samples, preprocess, model, device)

print(f"\nTrain: {train_embeddings.shape}  |  Val: {val_embeddings.shape}  |  Test: {test_embeddings.shape}")

# Save to Drive
emb_path = os.path.join(MODEL_DIR, "embeddings_maggots.npz")
np.savez(
    emb_path,
    train_embeddings=train_embeddings,
    train_labels    =train_labels,
    val_embeddings  =val_embeddings,
    val_labels      =val_labels,
    test_embeddings =test_embeddings,
    test_labels     =test_labels,
)
print(f"Saved embeddings -> {emb_path}")


In [ ]:
# =============================================================================
# 17. (OPTIONAL) RELOAD SAVED EMBEDDINGS
# =============================================================================
#
# Run this cell instead of Cell 17 if you just want to re-run the analysis
# without retraining the model.

emb_path = os.path.join(MODEL_DIR, "embeddings_maggots.npz")
data = np.load(emb_path)

train_embeddings = data["train_embeddings"]
train_labels     = data["train_labels"]
val_embeddings   = data["val_embeddings"]
val_labels       = data["val_labels"]
test_embeddings  = data["test_embeddings"]
test_labels      = data["test_labels"]

print(f"Loaded embeddings from {emb_path}")
print(f"Train: {train_embeddings.shape}  |  Val: {val_embeddings.shape}  |  Test: {test_embeddings.shape}")


In [ ]:
# =============================================================================
# 18. NEAREST NEIGHBOUR CLASSIFICATION  (KNN, k=5, cosine)
# =============================================================================

knn = KNeighborsClassifier(n_neighbors=5, metric='cosine')
knn.fit(train_embeddings, train_labels)

test_preds  = knn.predict(test_embeddings)
overall_acc = np.mean(test_preds == test_labels)

print(f"Overall KNN accuracy: {overall_acc*100:.1f}%")
print(f"\nPer-class accuracy:")

unique_labels = sorted(np.unique(test_labels))
target_names  = [aug_dataset.classes[i] for i in unique_labels]

for label in unique_labels:
    mask = test_labels == label
    acc  = np.mean(test_preds[mask] == test_labels[mask])
    n    = mask.sum()
    print(f"  [{label:2d}] {aug_dataset.classes[label]:45s}  {acc*100:.1f}%  ({int(acc*n)}/{n})")

print(f"\nClassification Report:")
print(classification_report(test_labels, test_preds, target_names=target_names))

In [ ]:
# =============================================================================
# 19. CONFUSION MATRIX
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 12))
cm   = confusion_matrix(test_labels, test_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=target_names)
disp.plot(ax=ax, colorbar=True, xticks_rotation=45)
ax.set_title("Confusion Matrix — EfficientNet-B0 + Triplet Loss (KNN)", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"confusion_matrix_cnn_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")


In [ ]:
# t-SNE plot (from saved embeddings, skip if running the whole pipeline)

emb_path = os.path.join(MODEL_DIR, "embeddings_flies.npz") # needs changing
data = np.load(emb_path, allow_pickle=True)

train_embeddings = data['train_embeddings']
train_labels     = data['train_labels']
test_embeddings  = data['test_embeddings']
test_labels      = data['test_labels']

all_embeddings = np.concatenate([train_embeddings, test_embeddings], axis=0)
all_labels     = np.concatenate([train_labels,     test_labels],     axis=0)
all_splits     = np.array(['train'] * len(train_embeddings) + ['test'] * len(test_embeddings))

unique_labels = sorted(np.unique(all_labels))
species_names = ['Chrysomya albiceps', 'Chrysomya marginalis', 'Chrysomya chloropyga', 'Lucilia sericata cuprina', 'Muscidae', 'Syntesiasomya nudiseta']
class_names = dict(zip(sorted(np.unique(all_labels)), species_names))

print("Computing t-SNE...")
tsne          = TSNE(n_components=2, perplexity=50, random_state=42, n_iter=1000)
embeddings_2d = tsne.fit_transform(all_embeddings)

colors = plt.colormaps['tab20'].resampled(len(unique_labels))

fig, ax = plt.subplots(figsize=(14, 10))
for i, label in enumerate(unique_labels):
    for split, marker, size in [('train', 'o', 40), ('test', '*', 120)]:
        mask = (all_labels == label) & (all_splits == split)
        ax.scatter(embeddings_2d[mask, 0], embeddings_2d[mask, 1],
                   color=colors(i), marker=marker, s=size, alpha=0.6,
                   label=class_names[label] if split == 'train' else None)

ax.set_title(f"t-SNE — EfficientNet-B0 ({MODE})\n(circles = train, stars = test)", fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"tsne_plot_cnn_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Drive.")


In [ ]:
# =============================================================================
# 20. t-SNE VISUALIZATION
# =============================================================================

# Combine all splits for a richer plot (same approach as BioCLIP notebook)
all_embeddings_combined = np.concatenate([train_embeddings, test_embeddings], axis=0)
all_labels_combined     = np.concatenate([train_labels,     test_labels],     axis=0)
all_splits              = np.array(
    ['train'] * len(train_embeddings) + ['test'] * len(test_embeddings)
)

print("Computing t-SNE (may take a few minutes)...")
tsne          = TSNE(n_components=2, perplexity=50, random_state=42, n_iter=1000)
embeddings_2d = tsne.fit_transform(all_embeddings_combined)

fig, ax = plt.subplots(figsize=(14, 10))
colors  = plt.colormaps['tab20'].resampled(len(unique_labels))

for i, label in enumerate(unique_labels):
    mask_train = (all_labels_combined == label) & (all_splits == 'train')
    mask_test  = (all_labels_combined == label) & (all_splits == 'test')

    ax.scatter(embeddings_2d[mask_train, 0], embeddings_2d[mask_train, 1],
               color=colors(i), marker='o', s=40, alpha=0.6, label=aug_dataset.classes[label])
    ax.scatter(embeddings_2d[mask_test,  0], embeddings_2d[mask_test,  1],
               color=colors(i), marker='*', s=120, alpha=0.9,
               edgecolors='black', linewidths=0.5)

ax.set_title(
    "t-SNE — EfficientNet-B0 Triplet Embeddings\n(circles = train, stars = test)",
    fontsize=13
)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"tsne_plot_cnn_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")


In [ ]:
# =============================================================================
# 21. TRAINING HISTORY PLOTS
# =============================================================================

epochs = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss
axes[0].plot(epochs, history["train_loss"], label="Train Loss", marker='o')
axes[0].plot(epochs, history["val_loss"],   label="Val Loss",   marker='o')
axes[0].set_title("Loss per Epoch"); axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Triplet Loss");  axes[0].legend(); axes[0].grid(True)

# Accuracy
axes[1].plot(epochs, [a * 100 for a in history["train_acc"]], label="Train Acc", marker='o')
axes[1].plot(epochs, [a * 100 for a in history["val_acc"]],   label="Val Acc",   marker='o')
axes[1].set_title("Accuracy per Epoch"); axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)");      axes[1].legend(); axes[1].grid(True)

# Mining stats
axes[2].plot(epochs, history["easy_pct"],      label="Easy",      marker='o')
axes[2].plot(epochs, history["semi_hard_pct"], label="Semi-Hard", marker='o')
axes[2].plot(epochs, history["hard_pct"],      label="Hard",      marker='o')
axes[2].set_title("Triplet Mining Statistics"); axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Percentage (%)");           axes[2].legend(); axes[2].grid(True)

plt.suptitle("Training History — EfficientNet-B0 + Triplet Loss", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"training_history_cnn_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved.")


In [ ]:
# =============================================================================
# 22. UMAP VISUALIZATION
# =============================================================================

import umap

print("Computing UMAP...")
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.1, metric ='cosine')
embeddings_2d_umap = reducer.fit_transform(all_embeddings_combined)

fig, ax = plt.subplots(figsize=(14, 10))
colors = plt.colormaps['tab20'].resampled(len(unique_labels))

for i, label in enumerate(unique_labels):
    for split, marker, size in [('train', 'o', 40), ('test', '*', 120)]:
        mask = (all_labels_combined == label) & (all_splits == split)
        # ax.scatter(embeddings_2d_umap[mask, 0], embeddings_2d_umap[mask, 1],
        #            color=colors(i), marker=marker, s=size, alpha=0.6,
        #            label=class_names[label] if split == 'train' else None)
        ax.scatter(embeddings_2d_umap[mask, 0], embeddings_2d_umap[mask, 1],
                   color=colors(i), marker=marker, s=size, alpha=0.6,
                   label=aug_dataset.classes[label] if split == 'train' else None)

ax.set_title("UMAP — BioCLIP + LoRA Embeddings\n(circles = train, stars = test)", fontsize=13)
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(PLOT_DIR, f"umap_plot_cnn_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()
print("Saved to Drive.")

In [ ]:
# =============================================================================
# 23. ROC CURVES + EQUAL ERROR RATE (EER)
# =============================================================================

from sklearn.metrics import roc_curve, auc

# Use the dataset's own mapping — this is the ground truth
idx_to_class = {v: k for k, v in aug_dataset.class_to_idx.items()}

unique_labels = sorted(np.unique(test_labels))
test_labels_bin = label_binarize(test_labels, classes=unique_labels)
test_probs = knn.predict_proba(test_embeddings)

n_classes = len(unique_labels)
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

eer_results = {}

for i in range(n_classes):
    fpr, tpr, thresholds = roc_curve(test_labels_bin[:, i], test_probs[:, i])
    roc_auc = auc(fpr, tpr)

    fnr = 1 - tpr
    eer_idx = np.argmin(np.abs(fpr - fnr))
    eer = (fpr[eer_idx] + fnr[eer_idx]) / 2
    eer_threshold = thresholds[eer_idx]

    # Correct mapping: loop index i → actual label → class name
    actual_label = int(knn.classes_[i])
    class_name = idx_to_class[actual_label]

    eer_results[class_name] = {
        "EER": round(eer, 4),
        "Threshold": round(float(eer_threshold), 4),
        "AUC": round(roc_auc, 4)
    }

    axes[i].plot(fpr, tpr, lw=2, label=f"AUC={roc_auc:.2f}")
    axes[i].scatter(fpr[eer_idx], tpr[eer_idx], color="red", zorder=5, label=f"EER={eer:.2f}")
    axes[i].plot([0, 1], [1, 0], "k--", lw=1, alpha=0.4)
    axes[i].plot([0, 1], [0, 1], "gray", lw=1, linestyle=":", alpha=0.4)
    axes[i].set_title(f"{class_name}\nAUC={roc_auc:.2f}  EER={eer:.2f}", fontsize=9)
    axes[i].set_xlabel("FPR")
    axes[i].set_ylabel("TPR")
    axes[i].set_xlim([0, 1])
    axes[i].set_ylim([0, 1])
    axes[i].legend(fontsize=7)
    axes[i].grid(True)

for j in range(n_classes, len(axes)):
    axes[j].set_visible(False)

plt.suptitle("ROC Curves — BioCLIP + LoRA (One-vs-Rest)", fontsize=14)
plt.tight_layout()
# plt.savefig(os.path.join(PLOT_DIR, f"roc_curve_vit16_{MODE}.png"), dpi=150, bbox_inches='tight')
plt.show()

# Print EER summary table
print(f"\n{'Class':<35} {'EER':>6} {'Threshold':>10} {'AUC':>6}")
print("-" * 60)
for class_name, metrics in eer_results.items():
    print(f"{class_name:<35} {metrics['EER']:>6.4f} {metrics['Threshold']:>10.4f} {metrics['AUC']:>6.4f}")

mean_eer = np.mean([v["EER"] for v in eer_results.values()])
print("-" * 60)
print(f"{'Mean EER':<35} {mean_eer:>6.4f}")


In [ ]:
# =============================================================================
# 24. FINAL EVALUATION: SEEN vs. UNSEEN (OOD DETECTION)
# =============================================================================

# 1. Define Unseen Samples (Loading from the UNSEEN_DIR)
unseen_samples = []
for root, dirs, files in os.walk(UNSEEN_DIR):
    for f in files:
        if f.lower().endswith(('.png', '.jpg', '.jpeg')):
            path = os.path.join(root, f)
            # Use a dummy label (0) since we only care about 'unseen' status
            unseen_samples.append((path, 0))

if len(unseen_samples) == 0:
    print(f"⚠️ WARNING: No images found in {UNSEEN_DIR}. Did you run the post-process cell?")

def extract_embeddings_final(samples, desc=""):
    """Helper to extract L2-normalized embeddings from the current model."""
    model.eval()
    embeddings, labels = [], []
    for path, label in tqdm(samples, desc=desc):
        try:
            img = Image.open(path).convert("RGB")
            img_t = preprocess(img).unsqueeze(0).to(device)
            with torch.no_grad():
                feat = model.encode_image(img_t)
                # L2 normalize for cosine similarity/distance
                feat = F.normalize(feat, dim=-1).squeeze().cpu().numpy()
            embeddings.append(feat)
            labels.append(label)
        except Exception as e:
            print(f"  ⚠️ Skipping {path}: {e}")

    return np.array(embeddings), np.array(labels)

# 2. Extract all necessary embeddings
# (Assumes train_samples and test_samples exist from your earlier dataset split)
print("Extracting current train embeddings (for KNN index)...")
train_embeddings_final, _ = extract_embeddings_final(train_samples, desc="Train")

print("\nExtracting current test embeddings (Seen classes)...")
test_embeddings_final, _ = extract_embeddings_final(test_samples, desc="Test")

print("\nExtracting unseen embeddings (Out-of-Distribution classes)...")
unseen_embeddings, _ = extract_embeddings_final(unseen_samples, desc="Unseen")

# 3. Balance Test and Unseen sets to equal size for fair F1 calculation
n_eval = min(len(test_embeddings_final), len(unseen_embeddings))

if n_eval == 0:
    raise ValueError("Error: Either Test or Unseen dataset is empty. Cannot perform evaluation.")

np.random.seed(42)
idx_test   = np.random.choice(len(test_embeddings_final), size=n_eval, replace=False)
idx_unseen = np.random.choice(len(unseen_embeddings),      size=n_eval, replace=False)

test_bal   = test_embeddings_final[idx_test]
unseen_bal = unseen_embeddings[idx_unseen]

print(f"\n✅ Balanced Evaluation: {n_eval} Seen vs {n_eval} Unseen")

# 4. Fit KNN and Query Distances
K = 5
knn = NearestNeighbors(n_neighbors=K, metric="cosine", algorithm="brute")
knn.fit(train_embeddings_final)

test_dists, _   = knn.kneighbors(test_bal)
unseen_dists, _ = knn.kneighbors(unseen_bal)

test_mean_dist   = test_dists.mean(axis=1)
unseen_mean_dist = unseen_dists.mean(axis=1)

# 5. Threshold Sweep for Anomaly Detection
all_dists = np.concatenate([test_mean_dist, unseen_mean_dist])
all_true  = np.array([1]*n_eval + [0]*n_eval) # 1=Seen, 0=Unseen

# Generate 300 possible thresholds based on actual distance ranges
thresholds = np.linspace(all_dists.min(), all_dists.max(), 300)
f1_scores, precisions, recalls, unseen_rejected = [], [], [], []

for thresh in thresholds:
    accepted = all_dists < thresh
    tp = ((accepted)  & (all_true == 1)).sum()
    fp = ((accepted)  & (all_true == 0)).sum()
    tn = ((~accepted) & (all_true == 0)).sum()
    fn = ((~accepted) & (all_true == 1)).sum()

    precision = tp / (tp + fp) if (tp + fp) > 0 else 1.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    f1_scores.append(f1)
    precisions.append(precision)
    recalls.append(recall)
    unseen_rejected.append(tn / (tn + fp) if (tn + fp) > 0 else 0.0)

best_idx    = np.argmax(f1_scores)
best_thresh = thresholds[best_idx]

print(f"\n🏆 Best OOD Threshold: {best_thresh:.4f}")
print(f"   F1 Score: {f1_scores[best_idx]*100:.1f}%")
print(f"   Unseen Rejected: {unseen_rejected[best_idx]*100:.1f}%")

# 6. Plots
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot A: Metrics vs Threshold
ax = axes[0]
ax.plot(thresholds, np.array(f1_scores)*100, label="F1", color="purple", lw=2.5)
ax.plot(thresholds, np.array(precisions)*100, label="Precision", color="steelblue", lw=1.5, linestyle="--")
ax.plot(thresholds, np.array(recalls)*100, label="Recall (Seen)", color="green", lw=1.5, linestyle="--")
ax.plot(thresholds, np.array(unseen_rejected)*100, label="Unseen Rejected %", color="tomato", lw=1.5, linestyle="--")
ax.axvline(best_thresh, color="black", linestyle=":", label=f"Best Thresh ({best_thresh:.3f})")
ax.set_title(f"Detection Performance vs. Distance\n(Mode: {MODE})")
ax.set_xlabel("Mean Cosine Distance (KNN)")
ax.set_ylabel("Percentage (%)")
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)

# Plot B: Distance Distribution
ax = axes[1]
ax.hist(test_mean_dist, bins=35, alpha=0.6, color="steelblue", label=f"Seen (Test)")
ax.hist(unseen_mean_dist, bins=35, alpha=0.6, color="tomato", label=f"Unseen (OOD)")
ax.axvline(best_thresh, color="black", linestyle=":", label="Decision Boundary")
ax.set_title(f"Distance Distributions: Seen vs Unseen\n(Model: EfficientNetB0)")
ax.set_xlabel(f"Mean Cosine Distance to {K} Neighbors")
ax.set_ylabel("Frequency")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
os.makedirs(PLOT_DIR, exist_ok=True)
save_path = os.path.join(PLOT_DIR, f"ood_analysis_efficientnetB0_{MODE}.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"✅ Final evaluation complete. Analysis saved to: {save_path}")

In [ ]:
# =============================================================================
# GRAD-CAM IMPLEMENTATION & SAVING DIRECTLY TO GOOGLE DRIVE (MAY BE UNFINISHED)
# =============================================================================
import os
import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from torchvision import transforms
import random

class CustomGradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None

        # Register PyTorch hooks
        self.forward_hook = target_layer.register_forward_hook(self.save_activation)
        self.backward_hook = target_layer.register_full_backward_hook(self.save_gradient)

    def save_activation(self, module, input, output):
        self.activations = output

    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0]

    def remove_hooks(self):
        self.forward_hook.remove()
        self.backward_hook.remove()

    def generate_cam(self, x, class_idx=None):
        self.model.eval()
        x.requires_grad_()

        out = self.model(x)
        if class_idx is None:
            class_idx = torch.argmax(out, dim=1).item()

        self.model.zero_grad()
        class_loss = out[0, class_idx]
        class_loss.backward(retain_graph=True)

        gradients = self.gradients.cpu().data.numpy()[0]
        activations = self.activations.cpu().data.numpy()[0]

        weights = np.mean(gradients, axis=(1, 2))

        cam = np.zeros(activations.shape[1:], dtype=np.float32)
        for i, w in enumerate(weights):
            cam += w * activations[i, :, :]

        cam = np.maximum(cam, 0)
        cam_max = np.max(cam)
        if cam_max > 0:
            cam = cam / cam_max

        return cam

# --- 1. SETUP & LOAD WEIGHTS ---
device = next(model.parameters()).device

# LOAD YOUR CUSTOM WEIGHTS HERE
# Ensure MODEL_DIR is defined, e.g., MODEL_DIR = '/content/drive/MyDrive/EffNet_models'
weights_path = f'{MODEL_DIR}/efficientnet_b0_triplet_best.pt'
if os.path.exists(weights_path):
    print(f"Loading weights from {weights_path}...")
    model.load_state_dict(torch.load(weights_path, map_location=device), strict=False)
    print("Weights loaded successfully!")
else:
    print(f"⚠️ WARNING: {weights_path} not found. Using the model's current weights in memory.")

model.eval()

# --- FETCH A BATCH AND HANDLE TRIPLET DATALOADER ---
batch = next(iter(val_loader))

if len(batch) == 3:
    images = batch[0]
    labels = torch.zeros(images.size(0), dtype=torch.long)
    print("Detected 3 items in batch. Using 'Anchor' images for Grad-CAM.")
elif len(batch) == 4:
    images = batch[0]
    labels = batch[3]
    print("Detected 4 items in batch. Using 'Anchor' images and provided labels.")
else:
    images = batch[0]
    labels = batch[-1]

num_images_to_save = min(10, images.size(0))
indices = random.sample(range(images.size(0)), num_images_to_save)

target_layers = {
    "Block_4_Early": model.features[4],
    "Block_6_Middle": model.features[6],
    "Block_8_Late": model.features[8]
}

inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

# --- SAVE DIRECTLY TO GOOGLE DRIVE ---
# This assumes you have already mounted your drive in a previous cell
base_output_dir = "/content/drive/MyDrive/gradcam_outputs"
os.makedirs(base_output_dir, exist_ok=True)

print(f"Saving Grad-CAM images to: {base_output_dir}/")

# --- 2. GENERATE AND SAVE LOOP ---
for img_count, img_idx in enumerate(indices):
    img_tensor = images[img_idx].unsqueeze(0).to(device)
    label = labels[img_idx].item()

    img_dir = os.path.join(base_output_dir, f"image_{img_count+1}_class_{label}")
    os.makedirs(img_dir, exist_ok=True)

    display_img = inv_normalize(images[img_idx]).permute(1, 2, 0).cpu().numpy()
    display_img = np.clip(display_img, 0, 1)

    plt.imsave(os.path.join(img_dir, "0_original.png"), display_img)

    for layer_name, layer in target_layers.items():
        cam_extractor = CustomGradCAM(model, layer)

        cam = cam_extractor.generate_cam(img_tensor, class_idx=label)
        cam_extractor.remove_hooks()

        cam_tensor = torch.tensor(cam).unsqueeze(0).unsqueeze(0)
        cam_resized = F.interpolate(cam_tensor, size=(display_img.shape[0], display_img.shape[1]),
                                    mode='bilinear', align_corners=False)
        cam_resized = cam_resized.squeeze().numpy()

        # Fixed the deprecation warning here:
        cmap = matplotlib.colormaps['jet']
        heatmap = cmap(cam_resized)[..., :3]

        overlay = 0.5 * heatmap + 0.5 * display_img
        overlay = np.clip(overlay, 0, 1)

        plt.imsave(os.path.join(img_dir, f"{layer_name}.png"), overlay)

    print(f"Saved Image {img_count+1}/{num_images_to_save} to {img_dir}/")

print("All Grad-CAM images have been successfully saved to your Google Drive!")